In [17]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv("../data/raw/CARGA_ENERGIA_2024.csv", sep=";", encoding="utf-8") #lê o arquivo CSV e armazena em um dataframe
df.head() #mostra as primeiras linhas do dataframe
df.info() #informações sobre o dataframe, como tipos de dados e valores nulos

#padronização dos nomes das colunas:
df.columns = (
    df.columns
    .str.strip() #remove espaços em branco
    .str.lower() #deixa tudo em minúsculo
    .str.replace(" ", "_") #substitui espaços por underline
)
print(df.columns.tolist())

In [ ]:
#tratamento de dados:
df["din_instante"] = pd.to_datetime(df["din_instante"]) #converte a coluna "din_instante" para o formato datetime
df["val_cargaenergiamwmed"] = pd.to_numeric(df["val_cargaenergiamwmed"]) #converte a coluna "val_cargaenergiamwmed" para o formato numérico
#verificar se há valores nulos:
print((df.isnull().sum()))
#não foram encontrados valores nulos, então segue a análise
print(df.duplicated().sum())
#não foram encontrados valores duplicados, então segue a análise

In [ ]:
#extração de informações de data:
df["din_instante_ano"] = df["din_instante"].dt.year
df["din_instante_mes"] = df["din_instante"].dt.month
df["din_instante_dia"] = df["din_instante"].dt.day

df.describe() #detecta outilers e fornece estatísticas descritivas
df = df.sort_values("din_instante") #ordena em series temporais
print(df.head())

In [ ]:
plt.figure(figsize=(14,5))

plt.plot(
    df["din_instante"],
    df["val_cargaenergiamwmed"]
)

plt.title("Carga de Energia ao Longo do Tempo")
plt.xlabel("Data")
plt.ylabel("Carga MWmed")

plt.show()

Com base no arquivo "carga_energia_pelo_tempo.png" gerado a partir da execução da célula acima é possível concluir que há variações de carga de energia ao longo do ano com perídos com maior uso que outros, além de variações diárias ao longo dos meses. 

In [ ]:
plt.figure(figsize=(10,5))

sns.boxplot(
    data=df,
    x="nom_subsistema",
    y="val_cargaenergiamwmed"
)

plt.title("Distribuição da Carga por Subsistema")
plt.xlabel("Subsistema")
plt.ylabel("Carga MWmed")

plt.show()

A partir do arquivo "distribuicao_da_carga_por_subsistema.png" se obteve as seguintes conclusões:
- foi visto que o subsistema SE apresenta maior mediana de carga de consumo com valor mínimo mais elevado
- o subsistema N teve os menores valores além de serem os mais concentrados
- os subsistemas NE e S têm valores similares com o subsistema S apresentando maior variação de valores além de outliers que indicam dias com cargas excepcionalmente altas 
- em todos os subsistemas foi analisado uma mediana bem centralizada o que indica boxplots relativamente simétricos 

In [ ]:
media_mensal = (
    df.groupby("din_instante_mes")["val_cargaenergiamwmed"]
    .mean()
)

plt.figure(figsize=(10,5))

media_mensal.plot(marker="o")

plt.title("Média Mensal da Carga Energética")
plt.xlabel("Mês")
plt.ylabel("Carga Média MWmed")

plt.grid()

plt.show()

A partir da observação do gráfico do arquivo "media_mensal_da_carga_energetica.png" é notório que a média mensal da carga energética possui uma sazonalidade, onde, os meses inicias do ano possuem uma média relativa mais alta enquanto os meses medianos apresentam uma tendência de queda na média, que volta a subir nos meses finais.

In [ ]:
plt.figure(figsize=(12,4))

sns.boxplot(
    x=df["val_cargaenergiamwmed"]
)

plt.title("Identificação de Outliers")

plt.show()

O gráfico do arquivo "identificacao_de_outliers.png" mostra um boxplot de valores globais, ou seja, considera todas as cargas de todos os meses. Com isso:
- é observado valores extremos acima dos valores padrão indicando picos de consumo 
- ademais, é visto que há um alongamento do terceiro quadrante indicando maior variação dos valores acima do padrão